In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("C:/Users/Voror/Projects/Personal/sep"))

from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT
from SIDER_dataset.libraries.feature_evaluation_methods import feature_variance_reduction_scores
from SIDER_dataset.libraries.utils import get_clus_path
%load_ext autoreload
%autoreload 2

In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths()
print(len(paths), "datasets")
paths

12 datasets


[{'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_cardiac.csv',
  'dataset_name': 'CPI+fingerprint_cardiac',
  'label_set': ['se_C0016382',
   'se_C0018799',
   'se_C0003811',
   'se_C0428977',
   'se_C0027051',
   'se_C0018790'],
  'features': 2147},
 {'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_high_freq.csv',
  'dataset_name': 'CPI+fingerprint_high_freq',
  'label_set': ['se_C0027497',
   'se_C0018681',
   'se_C0011603',
   'se_C0015230',
   'se_C0042963',
   'se_C0012833'],
  'features': 2147},
 {'dataset_path': 'C:\\Users\\Voror\\Projects\\Personal/sep/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_low_freq.csv',
  'dataset_name': 'CPI+fingerprint_low_freq',
  'label_set': ['se_C0020580',
   'se_C0267792',
   'se_C0018524',
   'se_C0031117',
   'se_C0035078',
   'se_C0011570'],
  'features': 2147},
 {'dataset

In [3]:
k = 10
random_state = 42
performances = []
ranking_criteria = "MDI"  # or "VAR"
include_original_features_options = [True, False]
training_algorithm = "Variance Reduction"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy", "RankingLoss", "MacroPrecision", "MacroRecall",
                 "MacroFOne"]
max_size = 5
cv_results = []

for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criteria}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            training_algorithm,
            "_".join(eval_criteria),
            str(max_size),
        ]
    )
    logging_path = f"XofN_filter_jaccard/logs/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:10] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        if ranking_criteria == "VAR":
            feature_rankings = feature_variance_reduction_scores(train_dataset[features],
                                                                 train_dataset[path["label_set"]])
        elif ranking_criteria == "MDI":
            feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])
        else:
            raise NotImplementedError

        XofN_groupings, avg_features, gen_XofN_time = generate_XofN_list_filter_jaccard(
            train_dataset,
            feature_rankings,
            max_size,
            path["label_set"],
            logger
        )
        if len(XofN_groupings) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = group_features(
                    train_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_test_dataset = group_features(
                    test_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_train_dataset.to_csv(f"XofN_filter_jaccard/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"XofN_filter_jaccard/tmp/test_dataset.csv", index=False)

                training_start = time.perf_counter()
                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "XofN_filter_jaccard/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"XofN_filter_jaccard/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            eval_criteria + ['nodes', 'leaves', 'groups',
                             'avg_group_features', 'gen_XofN_time', 'training_time']].mean().reset_index()
        print(averages)
        cv_results.append(averages)
        performances = []
# paths[0] - features[:10] desktop 0.23m
# laptop ??m
# desktop 24h


--- Running with label:'['se_C0016382', 'se_C0018799', 'se_C0003811', 'se_C0428977', 'se_C0027051', 'se_C0018790']' training_algorithm:'Variance Reduction' eval_criterion:'['averageAUROC', 'HammingLoss', 'SubsetAccuracy', 'RankingLoss', 'MacroPrecision', 'MacroRecall', 'MacroFOne']' max_size:'5' ---
Logs can be found in XofN_filter_jaccard/logs/CPI+fingerprint_cardiac_Variance Reduction_averageAUROC_HammingLoss_SubsetAccuracy_RankingLoss_MacroPrecision_MacroRecall_MacroFOne_5_logs.txt.

Fold 1/10 (CPI+fingerprint_cardiac 1/12)

generate_XofN_list -> Generating groupings based on variance reduction:variance reduction.


🔄 Processing features: 100%|██████████| 2147/2147 [19:58<00:00,  1.79feat/s] 


XofN groups with 2 features: 1
XofN groups with 5 features: 429
pruning: True, include_original_features: with_org, averageAUROC: 0.6721116, HammingLoss: 0.22542, SubsetAccuracy: 0.4964, RankingLoss: 0.19702, MacroPrecision: 0.49575, MacroRecall: 0.30507, MacroFOne: 0.37602, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6463832, HammingLoss: 0.30216, SubsetAccuracy: 0.28777, RankingLoss: 0.20016, MacroPrecision: 0.3723, MacroRecall: 0.5012, MacroFOne: 0.42561, 
pruning: True, include_original_features: no_org, averageAUROC: 0.6636196, HammingLoss: 0.20983, SubsetAccuracy: 0.48201, RankingLoss: 0.16573, MacroPrecision: 0.58885, MacroRecall: 0.25431, MacroFOne: 0.35278, 
pruning: False, include_original_features: no_org, averageAUROC: 0.6499695, HammingLoss: 0.31535, SubsetAccuracy: 0.31655, RankingLoss: 0.1942, MacroPrecision: 0.35706, MacroRecall: 0.50287, MacroFOne: 0.41424, 

Fold 2/10 (CPI+fingerprint_cardiac 1/12)

generate_XofN_list -> Generating groupings 

🔄 Processing features: 100%|██████████| 2147/2147 [20:16<00:00,  1.77feat/s] 


XofN groups with 2 features: 1
XofN groups with 5 features: 429
pruning: True, include_original_features: with_org, averageAUROC: 0.5616517, HammingLoss: 0.29616, SubsetAccuracy: 0.35971, RankingLoss: 0.23973, MacroPrecision: 0.40059, MacroRecall: 0.18563, MacroFOne: 0.2531, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5456472, HammingLoss: 0.36211, SubsetAccuracy: 0.20863, RankingLoss: 0.22648, MacroPrecision: 0.32319, MacroRecall: 0.32415, MacroFOne: 0.32355, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5222034, HammingLoss: 0.28897, SubsetAccuracy: 0.36691, RankingLoss: 0.23813, MacroPrecision: 0.38989, MacroRecall: 0.10577, MacroFOne: 0.16532, 
pruning: False, include_original_features: no_org, averageAUROC: 0.6023176, HammingLoss: 0.32254, SubsetAccuracy: 0.2518, RankingLoss: 0.21904, MacroPrecision: 0.4019, MacroRecall: 0.42468, MacroFOne: 0.4123, 

Fold 3/10 (CPI+fingerprint_cardiac 1/12)

generate_XofN_list -> Generating groupings 

...

XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.6008566, HammingLoss: 0.38258, SubsetAccuracy: 0.14394, RankingLoss: 0.28032, MacroPrecision: 0.46737, MacroRecall: 0.37794, MacroFOne: 0.4155, 
pruning: False, include_original_features: with_org, averageAUROC: 0.6021439, HammingLoss: 0.4154, SubsetAccuracy: 0.14394, RankingLoss: 0.26359, MacroPrecision: 0.44601, MacroRecall: 0.52384, MacroFOne: 0.48035, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5551545, HammingLoss: 0.40909, SubsetAccuracy: 0.15152, RankingLoss: 0.32104, MacroPrecision: 0.41747, MacroRecall: 0.30392, MacroFOne: 0.34974, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5770724, HammingLoss: 0.40152, SubsetAccuracy: 0.10606, RankingLoss: 0.28228, MacroPrecision: 0.45456, MacroRecall: 0.51722, MacroFOne: 0.48316, 

Fold 9/10 (fingerprint_mid_freq 12/12)

generate_XofN_list -> Generating groupings based on variance reduction:var

🔄 Processing features: 100%|██████████| 540/540 [02:59<00:00,  3.00feat/s]


XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.568211, HammingLoss: 0.41858, SubsetAccuracy: 0.19084, RankingLoss: 0.27263, MacroPrecision: 0.47959, MacroRecall: 0.37216, MacroFOne: 0.41814, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5640815, HammingLoss: 0.43257, SubsetAccuracy: 0.099237, RankingLoss: 0.26402, MacroPrecision: 0.46511, MacroRecall: 0.46679, MacroFOne: 0.46512, 
pruning: True, include_original_features: no_org, averageAUROC: 0.5861436, HammingLoss: 0.40712, SubsetAccuracy: 0.16794, RankingLoss: 0.27432, MacroPrecision: 0.49504, MacroRecall: 0.42336, MacroFOne: 0.45457, 
pruning: False, include_original_features: no_org, averageAUROC: 0.5857914, HammingLoss: 0.42875, SubsetAccuracy: 0.083969, RankingLoss: 0.26472, MacroPrecision: 0.47059, MacroRecall: 0.52897, MacroFOne: 0.4963, 

Fold 10/10 (fingerprint_mid_freq 12/12)

generate_XofN_list -> Generating groupings based on variance reduction:

🔄 Processing features: 100%|██████████| 540/540 [02:59<00:00,  3.00feat/s]


XofN groups with 5 features: 108
pruning: True, include_original_features: with_org, averageAUROC: 0.6210985, HammingLoss: 0.3715, SubsetAccuracy: 0.19847, RankingLoss: 0.23743, MacroPrecision: 0.49241, MacroRecall: 0.40669, MacroFOne: 0.44508, 
pruning: False, include_original_features: with_org, averageAUROC: 0.5968646, HammingLoss: 0.39567, SubsetAccuracy: 0.16031, RankingLoss: 0.22966, MacroPrecision: 0.47208, MacroRecall: 0.55767, MacroFOne: 0.5109, 
pruning: True, include_original_features: no_org, averageAUROC: 0.6674475, HammingLoss: 0.32188, SubsetAccuracy: 0.22901, RankingLoss: 0.22434, MacroPrecision: 0.57828, MacroRecall: 0.46781, MacroFOne: 0.51473, 
pruning: False, include_original_features: no_org, averageAUROC: 0.6285753, HammingLoss: 0.39186, SubsetAccuracy: 0.16031, RankingLoss: 0.25002, MacroPrecision: 0.47165, MacroRecall: 0.54005, MacroFOne: 0.50286, 
   pruning include_original_features               dataset  averageAUROC  \
0    False                    no_org  f

In [4]:
# All results
save_path = "XofN_filter_jaccard/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)
final_grouped_res.to_csv(save_path + "all_results.csv")
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,True,with_org,fingerprint_mid_freq,0.579408,0.388351,0.213932,0.262936,0.466620,0.358456,0.402762,165.0,83.0,108.0,5.000000,179.886424,1.331792
1,True,no_org,fingerprint_mid_freq,0.594248,0.370247,0.239695,0.265560,0.502976,0.402422,0.443636,156.2,78.6,108.0,5.000000,179.886424,0.687208
2,False,with_org,fingerprint_mid_freq,0.587997,0.415778,0.144137,0.248663,0.447138,0.517872,0.477923,874.4,437.7,108.0,5.000000,179.886424,1.331792
3,False,no_org,fingerprint_mid_freq,0.594928,0.411482,0.126701,0.249512,0.454223,0.541122,0.491675,890.4,445.7,108.0,5.000000,179.886424,0.687208
4,True,with_org,fingerprint_low_freq,0.514266,0.246514,0.369776,0.291297,0.107208,0.025261,0.016925,14.0,7.5,108.0,5.000000,179.298568,1.271389
5,True,no_org,fingerprint_low_freq,0.517035,0.245761,0.370523,0.291777,0.071242,0.024332,0.000000,13.4,7.2,108.0,5.000000,179.298568,0.660325
6,False,with_org,fingerprint_low_freq,0.589831,0.343824,0.190604,0.257230,0.329980,0.399092,0.357813,837.2,419.1,108.0,5.000000,179.298568,1.271389
7,False,no_org,fingerprint_low_freq,0.598470,0.343624,0.175400,0.252018,0.338613,0.428935,0.374190,850.6,425.8,108.0,5.000000,179.298568,0.660325
8,True,no_org,fingerprint_high_freq,0.571387,0.230251,0.501533,0.162637,0.794673,0.950802,0.864962,44.6,22.8,108.0,5.000000,178.951654,0.688364
9,False,no_org,fingerprint_high_freq,0.578008,0.276338,0.372346,0.161691,0.799650,0.859393,0.827641,599.2,300.1,108.0,5.000000,178.951654,0.688364


In [5]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_filter_jaccard/"
rounded_final_grouped_res = pd.read_csv(save_path + "all_results.csv", index_col=0)
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, eval_criteria, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,RankingLoss,MacroPrecision,MacroRecall,MacroFOne,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
1,no_org,fingerprint_mid_freq,0.594,0.370,0.240,0.266,0.503,0.402,0.444,156.2; 78.6,108.0; 5.0,0.0,179.9; 0.7
0,with_org,fingerprint_mid_freq,0.579,0.388,0.214,0.263,0.467,0.358,0.403,165.0; 83.0,108.0; 5.0,0.0,179.9; 1.3
5,no_org,fingerprint_low_freq,0.517,0.246,0.371,0.292,0.071,0.024,0.000,13.4; 7.2,108.0; 5.0,0.0,179.3; 0.7
4,with_org,fingerprint_low_freq,0.514,0.247,0.370,0.291,0.107,0.025,0.017,14.0; 7.5,108.0; 5.0,0.0,179.3; 1.3
8,no_org,fingerprint_high_freq,0.571,0.230,0.502,0.163,0.795,0.951,0.865,44.6; 22.8,108.0; 5.0,0.0,179.0; 0.7
11,with_org,fingerprint_high_freq,0.568,0.234,0.502,0.165,0.793,0.947,0.862,55.0; 28.0,108.0; 5.0,0.0,179.0; 1.3
13,no_org,fingerprint_cardiac,0.538,0.267,0.369,0.225,0.273,0.076,0.070,29.2; 15.1,108.0; 5.0,0.0,178.2; 0.7
12,with_org,fingerprint_cardiac,0.557,0.279,0.343,0.233,0.372,0.126,0.151,55.8; 28.4,108.0; 5.0,0.0,178.2; 1.3
17,no_org,CPI_mid_freq,0.600,0.368,0.240,0.260,0.538,0.362,0.428,108.2; 54.6,322.0; 5.0,0.0,836.5; 1.0
16,with_org,CPI_mid_freq,0.588,0.383,0.243,0.259,0.510,0.361,0.418,116.2; 58.6,322.0; 5.0,0.0,836.5; 2.8
